In [ ]:
import requests

url = "https://open.data.gov.sa/odp-public/267814ec-1d12-40f6-9e6a-a2b3239aef7c/d96dc4ce-c898-40cd-9b92-b5d4a1020016/v1/Labor"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/csv,*/*",
    "Referer": "https://open.data.gov.sa/",
}

r = requests.get(url, headers=headers, timeout=60)
print("status:", r.status_code)
print("content-type:", r.headers.get("content-type"))
r.raise_for_status()

with open("data.csv", "wb") as f:
    f.write(r.content)

print("saved data.csv, bytes:", len(r.content))


status: 200
content-type: text/html
saved data.csv, bytes: 245


**NOTE** I got this link - https://open.data.gov.sa/odp-public/267814ec-1d12-40f6-9e6a-a2b3239aef7c/d96dc4ce-c898-40cd-9b92-b5d4a1020016/v1/Labor - from the "Saudi Labor Office Resources" GET request - can be found on Postman

In [ ]:
import requests

url = "https://ksau-hs.edu.sa/qiyas/api/GetFacultyData/GetFacultyDataPublic"
#r = requests.get(url, timeout=30)
#print(r.status_code)
#print(r.headers.get("content-type"))
#print(r.text[:500])


This is from a dataset "KSAU Faculty Members Statistics" which is labelled as having a realtime API. The problem is that the it uses a type of TLS (Transport Layer Security) that causes an error to happen (as you can see from running this code). The funny thing is, when I simply copy the URL above to my browser, it gives me the data right away.

In [12]:
import sys
import requests
from typing import Any, Dict, List, Optional

# =========================
# Easy to change:
dataset_id = "e9fdd79e-ce84-4e5c-a797-ec1012c17ba9"
# =========================

BASE = "https://open.data.gov.sa/data/api"
VERSION = "-1"
TIMEOUT_S = 20


def get_json(url: str) -> Dict[str, Any]:
    try:
        r = requests.get(url, timeout=TIMEOUT_S)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Request failed: {url}\n{e}") from e
    except ValueError as e:
        raise RuntimeError(f"Response was not valid JSON: {url}") from e


def ensure_str(x: Any) -> str:
    return "" if x is None else str(x)


def print_kv(label: str, value: Any, indent: int = 2) -> None:
    pad = " " * indent
    print(f"{pad}{label}: {ensure_str(value)}")


def print_section(title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def safe_list(x: Any) -> List[Any]:
    return x if isinstance(x, list) else []


def main() -> None:
    # 1) Dataset info
    dataset_url = f"{BASE}/datasets?version={VERSION}&dataset={dataset_id}"
    ds = get_json(dataset_url)

    title_en = ds.get("titleEn")
    title_ar = ds.get("titleAr")
    desc_en = ds.get("descriptionEn")
    desc_ar = ds.get("descriptionAr")
    lang = ds.get("language")
    provider_en = ds.get("providerNameEn")
    provider_ar = ds.get("providerNameAr")
    update_frequency = ds.get("updateFrequency")
    time_period = ds.get("timePeriod")
    created_at = ds.get("createdAt")
    updated_at = ds.get("updatedAt")
    organization_id = ds.get("organizationId")

    print_section("DATASET SUMMARY")
    print_kv("Dataset ID", ds.get("id") or dataset_id)
    print_kv("Title (EN)", title_en)
    print_kv("Title (AR)", title_ar)
    print_kv("Description (EN)", desc_en)
    print_kv("Description (AR)", desc_ar)
    print_kv("Provider (EN)", provider_en)
    print_kv("Provider (AR)", provider_ar)
    print_kv("Language", lang)
    print_kv("Update Frequency", update_frequency)
    print_kv("Time Period", time_period)
    print_kv("Date Created", created_at)
    print_kv("Last Updated", updated_at)

    # Categories: print descriptionEn for each category
    categories = safe_list(ds.get("categories"))
    print("\n  Categories - \"Sub-fields\" (EN):")
    if not categories:
        print("    (none)")
    else:
        for i, c in enumerate(categories, 1):
            print(f"    {i}. {ensure_str(c.get('titleEn'))}")

    categories = safe_list(ds.get("categories"))
    print("\n  Categories - \"Sub-fields\" (AR):")
    if not categories:
        print("    (none)")
    else:
        for i, c in enumerate(categories, 1):
            print(f"    {i}. {ensure_str(c.get('titleAr'))}")

    # Tags: you said "name" field, but the payload shows nameEn/nameAr.
    # We'll prefer nameEn, fallback to nameAr, then id.
    tags = safe_list(ds.get("tags"))
    print("\n  Tags:")
    if not tags:
        print("    (none)")
    else:
        for i, t in enumerate(tags, 1):
            name = t.get("name")  # if API ever returns 'name'
            if not name:
                name = t.get("nameEn") or t.get("nameAr") or t.get("id")
            print(f"    {i}. {ensure_str(name)}")

    # 2) Dataset resources
    resources_url = f"{BASE}/datasets/resources?version={VERSION}&dataset={dataset_id}"
    res_payload = get_json(resources_url)
    resources = safe_list(res_payload.get("resources"))

    print_section("DATASET RESOURCES")
    if not resources:
        print("  (no resources found)")
    else:
        for idx, r in enumerate(resources, 1):
            fmt = r.get("format")
            r_id = r.get("id")
            r_descEn = r.get("descriptionEn")
            r_descAr = r.get("descriptionAr")

            print(f"\n  Resource {idx}")
            print_kv("Resource ID", r_id, indent=4)
            print_kv("Format", fmt, indent=4)
            if r_descEn:
                print_kv("Description (EN)", r_descEn, indent=4)
            if r_descAr:
                print_kv("Description (AR)", r_descAr, indent=4)

            # Columns:

            columns = safe_list(r.get("columns"))
            print("    Columns (names):")
            if not columns:
                print("      (none)")
            else:
                for j, col in enumerate(columns, 1):
                    print(f"      {j}. {ensure_str(col.get('name'))}")

    # 3) Organization info (using organizationId from dataset)
    if not organization_id:
        print_section("ORGANIZATION")
        print("  organizationId not found in dataset response; cannot fetch organization info.")
        return

    org_url = f"{BASE}/organizations?version={VERSION}&organization={organization_id}"
    org = get_json(org_url)

    print_section("ORGANIZATION SUMMARY")
    print_kv("Organization ID", org.get("id") or organization_id)
    print_kv("Name (EN)", org.get("nameEn"))
    print_kv("Name (AR)", org.get("nameAr"))
    print_kv("Description (EN)", org.get("descriptionEn"))
    print_kv("Description (AR)", org.get("descriptionAr"))
    print_kv("Website URL", org.get("websiteUrl"))
    print_kv("Address (EN)", org.get("addressEn"))
    print_kv("Address (AR)", org.get("addressAr"))
    print_kv("Type (EN)", org.get("typeEn"))
    print_kv("Type (AR)", org.get("typeAr"))

    datasets_list = safe_list(org.get("datasets"))
    print_kv("Number of datasets", len(datasets_list))

    # Optional: show first few dataset titles for sanity (commented out)
    # if datasets_list:
    #     print("\n  Example datasets (titleEn):")
    #     for i, d in enumerate(datasets_list[:10], 1):
    #         print(f"    {i}. {ensure_str(d.get('titleEn'))}")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"\nERROR: {e}", file=sys.stderr)
        sys.exit(1)



DATASET SUMMARY
  Dataset ID: e9fdd79e-ce84-4e5c-a797-ec1012c17ba9
  Title (EN): Academic Advising
  Title (AR): الإرشاد الأكاديمي
  Description (EN): The University Academic Advising Dataset at Jouf University for the Year 2025
  Description (AR): مجموعة بيانات الإرشاد الجامعي في جامعة الجوف لعام 2025
  Language: English
  Provider (EN): jouf university
  Provider (AR): جامعة الجوف
  Update Frequency: YEARLY
  Time Period: 2025-08-01 - 2025-12-31
  Date Created: 2025-12-26 20:37:47
  Last Updated: 2025-12-28 23:45:41

  Categories - "Sub-fields" (EN):
    1. Education and Training
    2. Education
    3. Information Technology
    4. Social Services
    5. Health
    6. Disabilities

  Categories - "Sub-fields" (AR):
    1. التعليم والتدريب
    2. التعليم
    3. تقنية المعلومات
    4. الخدمات الاجتماعية
    5. الصحة
    6. ذوي الاعاقة

  Tags:
    1. الإرشاد الأكاديمي
    2. جامعة الجوف
    3. إرشاد
    4. Academic Advising
    5. Advising
    6. Jouf University
    7. Jouf

DATASET 

The above is a code that extracts important information about **a dataset** from the **Saudi Open Data Portal** given it's **dataset ID**